# Clase 6 — Análisis espacial: construir variables territoriales

**Sistemas de Información Geográfica**
Especialización en Ciencias Sociales Computacionales — Universidad Nacional Guillermo Brown

| | |
|---|---|
| **Unidad del programa** | 6 — Análisis espacial |
| **Duración** | 3 horas |
| **Versión** | 2026.1 |
| **Docente** | Renzo Polo |
| **Licencia** | CC BY-SA 4.0 |

---

## 1. La pregunta de hoy

> ### ¿Qué tiene cada barrio alrededor, y a qué distancia?

Hasta acá aprendimos a conseguir datos (Clase 3), a ponerlos en el sistema de coordenadas
correcto (Clase 4) y a dibujarlos sin engañar (Clase 5). Pero en los tres casos el dato ya
venía hecho: alguien había contado los hogares con NBI y nosotros lo mapeábamos.

Hoy cambia el rol. Hoy **el dato lo fabricamos nosotros**.

Trabajamos sobre los dos barrios de la Ciudad de Buenos Aires que descargamos de OpenStreetMap
en la Clase 3 —**Recoleta** y **Villa Lugano**— y construimos para cada uno variables que no
existen en ninguna tabla: cuántas farmacias tiene por kilómetro cuadrado, qué porcentaje de su
superficie está a menos de 500 metros de un efector público de salud, a qué distancia queda ese
efector.

Esas variables se llaman **territoriales** porque no se observan: se **calculan** a partir de
la posición relativa de las cosas. Son las que van a necesitar para el trabajo final, y esta
clase es el repertorio de operaciones que las produce.

## 2. Objetivos de esta clase

Al terminar, deberías poder:

1. **Unir** una tabla sin geometría a una capa geográfica por una clave común.
2. **Transformar** geometrías: centroides, áreas de influencia, envolventes, simplificación y
   disolución, y saber para qué sirve cada una.
3. **Relacionar** dos capas por su posición con una unión espacial, y agregar el resultado.
4. **Superponer** capas con `overlay`: intersección, diferencia y unión.
5. **Medir** distancias y encontrar el elemento más cercano.
6. **Armar** una tabla de variables territoriales y exportarla.

**Cada operación se verifica con un mapa o un gráfico antes de pasar a la siguiente.** No
alcanza con que el código corra: hay que ver que el resultado es el que se buscaba.

## 3. Material de esta clase

Esta clase no tiene presentación: la teoría está acá, en la sección 4, y después va todo a la
práctica.

| Bloque | Qué retoma de clases anteriores |
|---|---|
| Unir por clave | Clase 3 — los datos de poblaciones.org |
| Transformar geometrías | Clase 2 — la simplificación y el efecto de la escala |
| Uniones espaciales | Clase 2 — el conteo de escuelas por celda y el MAUP |
| Superposición y distancias | Clase 4 — por qué medir exige un CRS proyectado |
| Densidades | Clase 5 — conteo, porcentaje y densidad responden preguntas distintas |

**Datos.** Todos salen del repositorio del curso:

| Archivo | Contenido | Fuente |
|---|---|---|
| `osm_barrios_limites.gpkg` | Contorno de Recoleta y Villa Lugano | OpenStreetMap (Clase 3) |
| `osm_amenities_barrios.gpkg` | 1.770 equipamientos con etiqueta `amenity` | OpenStreetMap (Clase 3) |
| `salud_barrios.gpkg` | Los 13 efectores públicos de salud de esos barrios | IGN (Clase 3) |
| `indicadores_hogares_departamentos_2022.gpkg` | 527 departamentos del Censo 2022 | INDEC (Clase 5) |
| `censo_grupos_edad.csv` | Población por grupo de edad de esos departamentos | poblaciones.org |

---

## 4. El repertorio del análisis espacial

Una **variable territorial** es un atributo de una unidad del espacio —un barrio, un radio
censal, un departamento— que **no viene en ningún registro** y que se obtiene poniendo dos
capas una encima de la otra.

"Población del barrio" no es una variable territorial: la contó el censo. "Cantidad de
farmacias del barrio" sí lo es: nadie la contó, pero tenemos la capa de barrios y la capa de
farmacias, y de la relación entre las dos sale el número.

### 4.1 Las cuatro familias de operación

Prácticamente todo el análisis espacial vectorial entra en estas cuatro familias. Cada una
tiene su bloque en esta notebook.

| Familia | Qué hace | Operaciones | Bloque |
|---|---|---|---|
| **Transformar** | Devuelve una geometría nueva a partir de otra | `centroid`, `buffer`, `convex_hull`, `envelope`, `simplify`, `union_all` | 7 |
| **Relacionar** | Empareja los registros de dos capas según su posición | `sjoin` con `within`, `intersects`, `contains` | 8 |
| **Superponer** | Calcula la geometría común, o la diferencia, entre dos capas | `overlay`: `intersection`, `difference`, `union` | 9 |
| **Medir** | Devuelve un número a partir de una o dos geometrías | `area`, `length`, `distance`, `sjoin_nearest` | 10 |

Hay una operación más que no es espacial, pero que va siempre primero porque sin ella no hay
con qué trabajar: **unir por clave** la tabla que trae los datos con la capa que trae la
geometría. Empezamos por ahí.

### 4.2 Adónde queremos llegar: la tabla de análisis

Todo el trabajo de hoy apunta a una tabla con esta forma:

| barrio | superficie_km2 | farmacias_por_km2 | cobertura_500m_perc | dist_salud_m |
|---|---|---|---|---|
| Recoleta | … | … | … | … |
| Villa Lugano | … | … | … | … |

**Una fila por unidad de análisis y una columna por variable.** Es la tabla que después entra
en un modelo, en un gráfico o en un mapa, y es la que van a tener que producir en el trabajo
final. Cada bloque de hoy le agrega una columna.

Dos reglas de la casa, que vamos a sostener toda la clase:

- **La unidad va en el nombre.** `superficie_km2`, no `superficie`. `dist_salud_m`, no
  `distancia`. Dentro de seis meses no vas a acordarte.
- **Lo relativo, no lo absoluto.** Recoleta tiene 6,9 km² y Villa Lugano 9,3. Comparar conteos
  crudos entre unidades de distinto tamaño es el error de la Clase 5, ahora del lado de la
  producción del dato y no del dibujo.

### 4.3 Por qué esto exige un CRS proyectado

Ésta es la clase donde la Clase 4 se cobra la deuda.

Casi todo lo de hoy —superficies, áreas de influencia de 500 metros, densidades por kilómetro
cuadrado, distancias— son **mediciones**. Y en coordenadas geográficas (EPSG:4326) la unidad es
el **grado**, que no es una unidad de longitud: un grado de longitud mide 111 km en el Ecuador
y 0 en el polo.

Un `buffer(500)` sobre datos en EPSG:4326 no da 500 metros: da 500 **grados**.

Para la Ciudad de Buenos Aires el sistema correcto es **EPSG:5347**, POSGAR 2007 faja 5, cuyo
meridiano central es −58,5° —la ciudad está en −58,4°—. Sus unidades son metros.

> ⚠️ **Regla de la clase:** antes de toda operación métrica, comprobar el CRS. Vamos a
> escribir esa comprobación como código, no como comentario.

---

## 5. Preparación

### ▶️ Las bibliotecas

In [ ]:
!pip install -q "geopandas==1.0.1" "mapclassify==2.8.1" "matplotlib==3.9.2" \
               "folium==0.17.0" "geopy==2.4.1"

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import folium

print("Bibliotecas listas")

### ▶️ Los dos sistemas de coordenadas

Definimos de entrada los dos que vamos a usar, y para qué sirve cada uno:

- **EPSG:4326** — coordenadas geográficas. Es como vienen los datos y es lo que entienden los
  mapas web. Se usa para **mostrar**.
- **EPSG:5347** — POSGAR 2007 faja 5, en metros. Se usa para **medir**.

In [ ]:
DATOS = "https://raw.githubusercontent.com/renzoepolo/sig-ciencias-sociales/main/datos/"

GEOGRAFICO = "EPSG:4326"   # para mostrar
METRICO    = "EPSG:5347"   # para medir: POSGAR 2007 faja 5, en metros

### ▶️ Las capas

Tres capas, las mismas durante toda la clase: los barrios, los equipamientos y los efectores
públicos de salud.

In [ ]:
barrios = gpd.read_file(DATOS + "osm_barrios_limites.gpkg")
equip   = gpd.read_file(DATOS + "osm_amenities_barrios.gpkg")
salud   = gpd.read_file(DATOS + "salud_barrios.gpkg")

for nombre, capa in [("barrios", barrios), ("equipamientos", equip), ("salud", salud)]:
    print(f"{nombre:15s} {len(capa):5d} registros   {capa.crs}")

### 👀 Antes de operar, mirar

La primera verificación visual de la clase, y la que más veces les va a salvar el análisis:
**abrir las capas y ver que estén donde tienen que estar**. Un CRS mal declarado se detecta acá
en dos segundos, y no tres bloques más adelante con un número absurdo.

In [ ]:
mapa = barrios.explore(color="grey", style_kwds=dict(fill=False, weight=3),
                       tiles="CartoDB positron", name="Barrios")

equip.explore(m=mapa, color="steelblue", marker_kwds=dict(radius=2),
              tooltip=["name", "amenity"], name="Equipamiento (OSM)")

salud.explore(m=mapa, color="crimson", marker_kwds=dict(radius=6),
              tooltip=["nombre", "tipo"], name="Efectores de salud (IGN)")

folium.LayerControl().add_to(mapa)
mapa

Los puntos caen adentro de los barrios y los barrios caen sobre la Ciudad de Buenos Aires. Las
tres capas vienen de fuentes distintas —dos de OSM, una del IGN— y están alineadas. Podemos
seguir.

### ▶️ Una función de comprobación

La vamos a llamar antes de cada medición. Son cuatro líneas y evitan el error más caro de todo
el curso.

In [ ]:
def comprobar_metrico(capa, nombre="la capa"):
    """Verifica que la capa esté en un CRS proyectado con unidades en metros."""
    assert capa.crs is not None, f"{nombre} no tiene CRS declarado"
    assert capa.crs.is_projected, f"{nombre} está en {capa.crs.name}: no se puede medir"
    unidad = capa.crs.axis_info[0].unit_name
    assert unidad == "metre", f"{nombre} mide en {unidad}, no en metros"
    print(f"✓ {nombre}: {capa.crs.name} — medición en metros")

### ▶️ Las versiones métricas

Reproyectamos una vez, al principio, y de ahí en más el sufijo `_m` dice de un vistazo con cuál
estamos trabajando. Es la convención del curso.

In [ ]:
barrios_m = barrios.to_crs(METRICO)
equip_m   = equip.to_crs(METRICO)
salud_m   = salud.to_crs(METRICO)

comprobar_metrico(barrios_m, "barrios_m")
comprobar_metrico(equip_m, "equip_m")
comprobar_metrico(salud_m, "salud_m")

---

## 6. Unir una tabla a una geometría

### 🧭 Concepto

La situación es la de siempre: tenés una capa con las geometrías —departamentos, radios,
barrios— y, aparte, una planilla con los datos que te interesan. Ninguna de las dos sirve sola.

La operación que las junta es un **join por clave** (`merge` de pandas): se elige una columna
que esté en las dos tablas y que identifique unívocamente a cada unidad, y pandas empareja fila
con fila. En la Argentina esa clave suele ser el **código del INDEC**: cinco dígitos, dos de
provincia y tres de departamento. La Comuna 1 de la Ciudad es `02007`.

Éste es el único bloque que sale del escenario de los barrios: usamos los 527 departamentos del
país, porque es donde la operación se ve con claridad y porque es el dato que la mayoría va a
tener que unir en el trabajo final.

### ▶️ Las dos piezas

Del lado de la geometría, los departamentos de la Clase 5. Del lado de la tabla, la población
por grupo de edad, que el GeoPackage no trae.

In [ ]:
deptos = gpd.read_file(DATOS + "indicadores_hogares_departamentos_2022.gpkg")
edades = pd.read_csv(DATOS + "censo_grupos_edad.csv", dtype={"codigo": str})

print("Departamentos (geometría):", len(deptos), "filas")
print("Grupos de edad (tabla):   ", len(edades), "filas")
edades.head(3)

> 📌 **El único detalle que hay que cuidar:** `dtype={"codigo": str}`.
>
> El código del INDEC empieza con cero —`02007`— y si pandas lo lee como número se lo come y
> queda `2007`, que no coincide con nada. **Un código de identificación es una etiqueta, no un
> número:** no se suma, no se promedia, y se lee como texto. Vale igual para el CUIT, el código
> postal y el CUE de las escuelas.

### ▶️ La unión

`validate="1:1"` le pide a pandas que verifique que cada fila de un lado tenga a lo sumo una del
otro. Si la clave estuviera repetida, corta acá en lugar de multiplicar filas en silencio.

In [ ]:
deptos = deptos.merge(edades[["codigo", "poblacion_65_mas"]],
                      on="codigo", how="left", validate="1:1")

deptos["perc_65_mas"] = deptos["poblacion_65_mas"] / deptos["poblacion"] * 100

deptos[["codigo", "departamento", "provincia", "poblacion", "perc_65_mas"]].head()

### ✅ Comprobación

Un `merge` con `how="left"` nunca falla: los que no encuentran pareja quedan en blanco. Por eso
se cuentan, siempre, antes de seguir.

In [ ]:
sin_pareja = deptos["perc_65_mas"].isna().sum()

print(f"Filas: {len(deptos)} (esperadas: 527)")
print(f"Sin coincidencia: {sin_pareja}")
assert sin_pareja == 0, "Hay departamentos sin dato: revisar la clave"

### 👀 Y la comprobación que de verdad convence: el mapa

El `assert` dice que no hay nulos. El mapa lo **muestra**, y muestra algo que el número no
puede: si el join hubiera fallado para una provincia entera —porque su código venía con otro
formato, por ejemplo—, se vería un hueco blanco con forma de provincia.

**Un mapa de la variable recién unida es la mejor verificación de un join por clave.**

In [ ]:
fig, eje = plt.subplots(figsize=(6, 9))

deptos.plot(column="perc_65_mas", cmap="Purples", scheme="FisherJenks", k=5,
            legend=True, edgecolor="white", linewidth=0.1, ax=eje,
            missing_kwds=dict(color="red", label="Sin dato"),
            legend_kwds=dict(title="% de 65 años o más", loc="lower left"))

eje.set_title("Población de 65 años o más, 2022", fontsize=13)
eje.set_axis_off()
plt.show()

### 🔍 Interpretación

Ni un solo departamento en rojo: el join cubrió los 527.

Y el mapa ya dice algo. El sur de la provincia de Buenos Aires, La Pampa y el centro del país
aparecen oscuros —poblaciones envejecidas, con emigración de jóvenes—, y el norte y la Patagonia
aparecen claros.

El departamento más envejecido del país es la **Comuna 2 de la Ciudad de Buenos Aires**, con el
**20,3 %**, contra un promedio nacional del 11,4 %. La Comuna 2 **es** el barrio de Recoleta:
los dos límites coinciden. En el otro extremo de la ciudad, la Comuna 8 —donde está Villa
Lugano— tiene el 11,1 %, casi la mitad.

Guardate ese contraste, porque los bloques que siguen lo van a cruzar con la oferta de salud.

In [ ]:
deptos.loc[deptos["codigo"].isin(["02014", "02056"]),
           ["codigo", "departamento", "poblacion", "perc_65_mas"]]

---

## 7. Transformar geometrías

### 🧭 Concepto

La primera familia de operaciones toma una geometría y devuelve **otra geometría**. Todavía no
mide nada: prepara el terreno para medir.

| Operación | Qué devuelve | Para qué se usa |
|---|---|---|
| `centroid` | El centro de masa | Reducir un polígono a un punto para medir distancias |
| `representative_point()` | Un punto **garantizado adentro** | Lo mismo, cuando el polígono es cóncavo o tiene islas |
| `buffer(d)` | La zona a menos de *d* de la geometría | Áreas de influencia, zonas de servicio |
| `convex_hull` | La envolvente convexa: la "banda elástica" | Extensión de un conjunto de puntos |
| `envelope` | El rectángulo que la contiene | Recortes rápidos, vistas previas |
| `simplify(t)` | La misma forma con menos vértices | Aligerar mapas web |
| `union_all()` | Todo fundido en una geometría | Eliminar superposiciones antes de medir |

Todas trabajan en las unidades del CRS de la capa. De nuevo: **métrico**.

### ▶️ De polígono a punto

Dos formas de reducir un barrio a un punto, y no son equivalentes.

In [ ]:
centroides     = barrios_m.centroid                   # centro de masa
puntos_internos = barrios_m.representative_point()    # un punto cualquiera, pero adentro

pd.DataFrame({
    "barrio": barrios_m["barrio"],
    "centroide_adentro": centroides.within(barrios_m.geometry),
    "punto_interno_adentro": puntos_internos.within(barrios_m.geometry),
})

### 👀 Dónde cae cada uno

In [ ]:
fig, eje = plt.subplots(figsize=(7, 7))

barrios_m.plot(ax=eje, facecolor="#eef3f8", edgecolor="#2b6ca3", linewidth=1.5)
centroides.plot(ax=eje, color="crimson", markersize=90, label="centroid")
puntos_internos.plot(ax=eje, color="darkgreen", marker="x", markersize=90,
                     label="representative_point")

eje.legend(loc="upper right")
eje.set_title("Dos maneras de reducir un polígono a un punto")
eje.set_axis_off()
plt.show()

### 🔍 Interpretación

Acá los dos caen adentro, porque Recoleta y Villa Lugano son razonablemente compactos, y están
casi pegados.

Pero el centroide es el **centro de masa**, y nada garantiza que caiga dentro de la figura: el
centroide de una medialuna cae en el aire, y el de una provincia con una gran bahía puede caer
en el agua. Cuando lo que se necesita es *un punto cualquiera adentro* —para poner una etiqueta,
para muestrear—, `representative_point()` lo garantiza. Cuando lo que se necesita es *el
centro*, el centroide es el correcto.

> ⚠️ Y una advertencia que vale para todo el bloque: el centroide de un barrio **no es donde
> vive la gente**. Es una simplificación geométrica, no demográfica. Sirve para medir distancias
> aproximadas entre unidades; no sirve para afirmar a qué distancia vive la población.

### ▶️ Envolventes, simplificación y áreas de influencia

Las otras cuatro, sobre la misma capa, para poder compararlas con una sola mirada.

In [ ]:
formas = {
    "Original":       barrios_m.geometry,
    "Buffer 300 m":   barrios_m.buffer(300),
    "Convex hull":    barrios_m.convex_hull,
    "Envelope":       barrios_m.envelope,
    "Simplify 200 m": barrios_m.simplify(200),
}

pd.DataFrame({
    "superficie_km2": {k: round(v.area.sum() / 1e6, 2) for k, v in formas.items()},
    "vertices":       {k: int(v.count_coordinates().sum()) for k, v in formas.items()},
})

### 👀 Las cinco formas, una al lado de la otra

In [ ]:
fig, ejes = plt.subplots(1, 5, figsize=(18, 4))

for eje, (titulo, geom) in zip(ejes, formas.items()):
    gpd.GeoSeries(geom, crs=METRICO).plot(ax=eje, facecolor="#cfe3f5", edgecolor="#2b6ca3")
    barrios_m.plot(ax=eje, facecolor="none", edgecolor="black", linewidth=0.6)
    eje.set_title(titulo, fontsize=11)
    eje.set_axis_off()

plt.tight_layout()
plt.show()

El contorno negro es siempre el original, para poder ver qué le hizo cada operación.

### 👀 Qué se pierde al simplificar

La tabla dice que `simplify(200)` conserva casi toda la superficie. Pero lo que importa no es
cuánta superficie queda: es **dónde** cambió la forma. Un acercamiento al borde lo muestra.

In [ ]:
fig, eje = plt.subplots(figsize=(9, 7))

barrios_m.plot(ax=eje, facecolor="none", edgecolor="black", linewidth=1.6, label="Original")
gpd.GeoSeries(barrios_m.simplify(200), crs=METRICO).plot(
    ax=eje, facecolor="none", edgecolor="crimson", linewidth=1.6, linestyle="--")

eje.set_title("Original (negro) y simplificado a 200 m (rojo)")
eje.set_axis_off()
plt.show()

### 🔍 Interpretación — cada forma sirve para algo distinto

**`simplify(200)`** baja de 797 a 40 vértices —un 95 % menos— y pierde apenas el 3 % de la
superficie. Ésa es la operación que hace que un mapa web cargue: la Clase 5 la usó para que
Folium no se trabara con los 527 departamentos. El parámetro es la tolerancia en metros: cuánto
se permite que la línea nueva se aparte de la original. En el acercamiento se ve que las curvas
suaves se vuelven rectas, y que los detalles menores a 200 m desaparecen.

**`convex_hull`** agrega un 59 % de superficie en Recoleta. No es un error: la envolvente
convexa responde *"¿hasta dónde llega este conjunto?"*, no *"¿qué forma tiene?"*. Sirve para
delimitar el área que abarca un conjunto de puntos —el alcance de un operativo, la extensión de
una muestra—, no para reemplazar un límite administrativo.

**`envelope`** duplica la superficie. Es útil como recorte grosero y rápido, y para nada más.

**`buffer`** es la que vamos a usar en serio en el bloque 9.

---

## 8. Relacionar capas: uniones espaciales

### 🧭 Concepto

La **unión espacial** (`gpd.sjoin`) es el `merge` de la geografía: en lugar de emparejar filas
por una columna en común, las empareja por **posición**.

No hace falta que las dos capas compartan ningún atributo. Alcanza con que compartan el espacio
y con declarar la relación que buscamos, el **predicado**:

| Predicado | Empareja cuando… | Uso típico |
|---|---|---|
| `within` | la geometría de la izquierda está adentro de la de la derecha | puntos en polígonos |
| `intersects` | se tocan o se superponen, aunque sea en un borde | el más laxo, y el que viene por defecto |
| `contains` | la de la izquierda contiene a la de la derecha | polígonos que encierran puntos |

> ⚠️ El requisito que se olvida: las dos capas tienen que estar **en el mismo CRS**.

### ▶️ Unión 1 a 1 — a qué barrio pertenece cada punto

El caso simple: cada punto cae en un polígono y en uno solo, y se lleva sus atributos.

La capa de equipamientos ya trae una columna `barrio`, de cuando la bajamos en la Clase 3
consultando un barrio por vez. La sacamos para recalcularla: si quedaran las dos, GeoPandas
renombraría ambas a `barrio_left` y `barrio_right`, que es la fuente de la mitad de los enredos
con uniones.

In [ ]:
equip_m = equip_m.drop(columns=["barrio"])

equip_en_barrio = gpd.sjoin(equip_m, barrios_m[["barrio", "geometry"]], predicate="within")
equip_en_barrio = equip_en_barrio.drop(columns="index_right")

print(f"De {len(equip_m)} equipamientos, {len(equip_en_barrio)} cayeron dentro de un barrio")
equip_en_barrio[["name", "amenity", "barrio"]].head()

### 👀 Verificación — cada punto con el color del barrio que le tocó

Si el `sjoin` funcionó, cada punto tiene que estar pintado del color del polígono que lo
contiene. Un punto azul dentro del barrio rojo delataría un problema de CRS o de predicado al
instante.

In [ ]:
fig, eje = plt.subplots(figsize=(9, 8))

barrios_m.plot(ax=eje, facecolor="none", edgecolor="black", linewidth=1.5)
equip_en_barrio.plot(ax=eje, column="barrio", markersize=6, legend=True,
                     cmap="Set1", legend_kwds=dict(loc="upper right"))

eje.set_title("Cada equipamiento, con el barrio que le asignó el sjoin")
eje.set_axis_off()
plt.show()

### ✅ Comprobación

In [ ]:
perdidos = len(equip_m) - len(equip_en_barrio)

print(f"Equipamientos sin barrio asignado: {perdidos}")
print(equip_en_barrio["barrio"].value_counts().to_string())

Uno solo quedó afuera, de 1.770. Es un punto que cae exactamente sobre el límite: `within` exige
estar **estrictamente adentro**, y con `intersects` habría entrado. No es un error; es la
definición del predicado. Vale saber que esa diferencia existe y que, en un análisis con muchas
unidades vecinas, elegir `intersects` puede hacer que un punto se cuente dos veces.

### ▶️ Unión 1 a muchos — contar y agregar

Ahora al revés: nos interesa el **barrio**, y cada barrio contiene muchos puntos. El `sjoin` es
el mismo; lo que cambia es que después **agregamos** con `groupby`.

In [ ]:
conteo = equip_en_barrio.groupby("barrio").size().rename("equipamientos")

por_tipo = equip_en_barrio.pivot_table(index="amenity", columns="barrio",
                                       aggfunc="size", fill_value=0)

por_tipo.sort_values("Recoleta", ascending=False).head(10)

### ▶️ De conteo a densidad

Los conteos no son comparables: Recoleta tiene 6,9 km² y Villa Lugano 9,3. Es el problema de la
Clase 5, ahora del lado de la producción del dato.

La superficie la calculamos nosotros —otra variable territorial— con el CRS ya comprobado.

In [ ]:
barrios_m["superficie_km2"] = barrios_m.area / 1_000_000

densidad = por_tipo.T.join(barrios_m.set_index("barrio")["superficie_km2"])
for tipo in ["pharmacy", "school", "cafe", "bank", "clinic"]:
    densidad[tipo + "_por_km2"] = (densidad[tipo] / densidad["superficie_km2"]).round(1)

densidad[[c for c in densidad.columns if c.endswith("_por_km2")]]

### 👀 La comparación, en barras

La tabla tiene el dato; el gráfico tiene la conclusión.

In [ ]:
columnas = [c for c in densidad.columns if c.endswith("_por_km2")]
grafico = densidad[columnas].rename(columns=lambda c: c.replace("_por_km2", ""))

eje = grafico.T.plot.barh(figsize=(9, 5), color=["#c44e52", "#4c72b0"], width=0.75)
eje.set_xlabel("Equipamientos por km²")
eje.set_ylabel("")
eje.set_title("Densidad de equipamiento por barrio")
eje.legend(title="")
eje.grid(axis="x", linestyle=":", alpha=0.6)
plt.tight_layout()
plt.show()

### 🔍 Interpretación — dos lógicas de localización

El gráfico separa las barras en dos grupos sin que nadie se lo haya pedido.

**Farmacias: 12,4 por km² en Recoleta contra 0,8 en Villa Lugano —quince veces más.** Bancos:
8,2 contra 0,6. Cafés: 31,4 contra 0,4.

**Escuelas: 9,2 y 9,2. Idénticas.**

Las farmacias, los bancos y los cafés son equipamiento **de mercado**: se instalan donde hay
capacidad de compra. Las escuelas son equipamiento **del Estado**: se instalan donde hay
población en edad escolar. Dos lógicas de localización, dos huellas territoriales distintas, y
la misma tabla las distingue.

Y la fila de las escuelas hace de control de calidad del dato: si OpenStreetMap estuviera
subregistrando Villa Lugano de manera masiva, las escuelas también aparecerían subcontadas. Dan
iguales, así que la brecha de farmacias no es un artefacto de la fuente.

---

## 9. Superponer capas: intersección y diferencia

### 🧭 Concepto

El `sjoin` relaciona **registros**: le dice a cada punto en qué barrio está, pero no modifica
ninguna geometría. El **overlay** trabaja sobre las **geometrías**: corta unas contra otras y
devuelve las piezas.

| Operación | Devuelve |
|---|---|
| `intersection` | La parte común a las dos capas |
| `difference` | Lo que está en la primera y no en la segunda |
| `union` | Todas las piezas: lo común y lo propio de cada una |

Es la herramienta que responde *"¿qué parte de esta unidad está dentro de aquella zona?"*, y de
ahí sale una de las variables territoriales más usadas: el **porcentaje de cobertura**.

Nuestra pregunta concreta: **¿qué parte de cada barrio está a menos de 500 metros de un efector
público de salud?**

Los 500 metros son una decisión, no un dato: es una caminata de unos 6 minutos, el umbral que
suele usarse para servicios de proximidad. Al final del bloque vamos a ver cuánto dependía el
resultado de esa elección.

### ▶️ Paso 1 — El área de influencia

Acá la comprobación de CRS deja de ser una formalidad: `buffer(500)` toma el número en las
unidades del sistema de la capa.

In [ ]:
comprobar_metrico(salud_m, "salud_m")

RADIO_M = 500
areas = salud_m.copy()
areas["geometry"] = salud_m.buffer(RADIO_M)

print(f"{len(areas)} áreas de influencia de {RADIO_M} m")
print(f"Superficie de cada una: {areas.area.iloc[0] / 10_000:.1f} ha")

### 👀 Paso 2 — Ver el problema antes de medirlo

In [ ]:
fig, eje = plt.subplots(figsize=(9, 8))

barrios_m.plot(ax=eje, facecolor="#f2f2f2", edgecolor="black", linewidth=1.5)
areas.plot(ax=eje, facecolor="orange", alpha=0.35, edgecolor="darkorange", linewidth=0.8)
salud_m.plot(ax=eje, color="crimson", markersize=35)

eje.set_title("Áreas de influencia de 500 m — se superponen")
eje.set_axis_off()
plt.show()

En Villa Lugano los círculos **se superponen**: hay zonas cubiertas por dos o tres efectores a
la vez. Si sumáramos las superficies de los 13 círculos, esa zona contaría varias veces y el
porcentaje podría pasarse del 100 %.

### ▶️ Paso 3 — Disolver

`union_all()` funde todas las geometrías en una sola y elimina las superposiciones. Es la
operación que evita la doble contabilidad.

In [ ]:
cobertura = areas.union_all()

suma_circulos = areas.area.sum() / 1e6
area_disuelta = cobertura.area / 1e6

print(f"Suma de los 13 círculos: {suma_circulos:.2f} km²")
print(f"Área realmente cubierta: {area_disuelta:.2f} km²")
print(f"Se contaba de más:       {suma_circulos - area_disuelta:.2f} km²")

### ▶️ Paso 4 — Intersección y diferencia

Ahora el overlay propiamente dicho. Para cada barrio, la parte cubierta y la parte que queda
afuera.

In [ ]:
barrios_m["area_cubierta_km2"] = barrios_m.geometry.intersection(cobertura).area / 1e6
barrios_m["cobertura_500m_perc"] = (barrios_m["area_cubierta_km2"]
                                    / barrios_m["superficie_km2"] * 100)

barrios_m[["barrio", "superficie_km2", "area_cubierta_km2", "cobertura_500m_perc"]].round(1)

### 👀 Paso 5 — El mapa que responde la pregunta

Verde lo que está cubierto, rojo lo que no. Es la `intersection` y la `difference`, dibujadas
juntas.

In [ ]:
cubierto = barrios_m.copy()
cubierto["geometry"] = barrios_m.geometry.intersection(cobertura)

afuera = barrios_m.copy()
afuera["geometry"] = barrios_m.geometry.difference(cobertura)

fig, eje = plt.subplots(figsize=(9, 8))
cubierto.plot(ax=eje, facecolor="#7fbf7b", edgecolor="none")
afuera.plot(ax=eje, facecolor="#e08a8a", edgecolor="none")
barrios_m.plot(ax=eje, facecolor="none", edgecolor="black", linewidth=1.5)
salud_m.plot(ax=eje, color="darkred", markersize=30)

eje.set_title("A menos de 500 m de un efector (verde) y a más (rojo)")
eje.set_axis_off()
plt.show()

### ✅ Comprobación

Dos controles. El porcentaje tiene que estar entre 0 y 100, y las dos piezas del overlay tienen
que sumar exactamente el barrio entero: si `intersection` y `difference` no reconstruyen el
total, algo se perdió por el camino.

In [ ]:
p = barrios_m["cobertura_500m_perc"]
assert p.between(0, 100).all(), "Porcentaje fuera de rango: ¿se disolvieron los buffers?"

suma = (cubierto.area + afuera.area) / 1e6
print("Cubierto + sin cubrir vs. total del barrio (km²):")
print(pd.DataFrame({"reconstruido": suma.round(4),
                    "original": barrios_m["superficie_km2"].round(4)}).to_string(index=False))

### 👀 Paso 6 — ¿Y si el umbral fuera otro?

Los 500 metros los elegimos nosotros. Antes de concluir, conviene ver cuánto dependía el
resultado de esa elección.

In [ ]:
filas = []
for radio in [200, 300, 500, 750, 1000, 1500]:
    zona = salud_m.buffer(radio).union_all()
    for _, b in barrios_m.iterrows():
        filas.append(dict(radio_m=radio, barrio=b["barrio"],
                          cobertura=100 * b.geometry.intersection(zona).area / b.geometry.area))

sensibilidad = pd.DataFrame(filas).pivot(index="radio_m", columns="barrio", values="cobertura")
sensibilidad.round(1)

In [ ]:
eje = sensibilidad.plot(marker="o", figsize=(8, 5), color=["#c44e52", "#4c72b0"])
eje.set_xlabel("Radio del área de influencia (m)")
eje.set_ylabel("% de la superficie del barrio cubierta")
eje.set_title("La cobertura según el umbral elegido")
eje.grid(linestyle=":", alpha=0.6)
eje.legend(title="")
plt.tight_layout()
plt.show()

### 🔍 Interpretación — el resultado que no esperábamos

A 500 metros, **Villa Lugano tiene el 49 % de su superficie cubierta y Recoleta el 24 %**. El
doble.

Vale la pena detenerse, porque es lo contrario de lo que sugería el bloque anterior. En
farmacias, bancos y cafés Recoleta sacaba entre quince y ochenta veces de ventaja. En cobertura
de **salud pública de proximidad**, Villa Lugano gana por el doble.

Los dos resultados son correctos y hablan de cosas distintas:

- Recoleta tiene **cuatro** efectores públicos, y son **hospitales**: grandes, de alta
  complejidad, que atienden a toda la ciudad y no al barrio. Además es la comuna más envejecida
  del país —el 20,3 % del bloque 6—, o sea la población que más usa el sistema de salud.
- Villa Lugano tiene **nueve**, y son **CeSAC**: chicos, de primer nivel, pensados justamente
  para la atención de proximidad y colocados donde el mercado no pone nada.

**Y el gráfico de sensibilidad es lo que convierte esto en una conclusión defendible:** las dos
curvas no se cruzan en ningún punto. Villa Lugano va adelante a 200, a 500 y a 1.500 metros. El
resultado no depende de haber elegido bien el radio.

> ⚠️ Lo que la variable mide, y lo que no: la cobertura es de **superficie**, no de población, y
> "estar cerca" es una dimensión del acceso, no todas. Acota la conclusión; no la anula.

---

## 10. Medir distancias

### 🧭 Concepto

La cobertura contesta con un sí o un no: adentro o afuera de los 500 metros. La **distancia**
contesta con un número, y por eso es la variable territorial más informativa de las cuatro.

| Operación | Qué hace |
|---|---|
| `a.distance(b)` | Distancia mínima entre dos geometrías, elemento a elemento |
| `gpd.sjoin_nearest(a, b)` | Para cada fila de `a`, la fila **más cercana** de `b` |
| `distance_col=` | Guarda esa distancia en una columna |

`sjoin_nearest` es la operación estrella del bloque: resuelve de una sola vez "¿cuál es el más
cercano y a qué distancia está?", que a mano requeriría comparar todos contra todos.

> ⚠️ Es distancia **en línea recta**. La distancia caminando por las calles es mayor, siempre, y
> la vamos a calcular en la Clase 7.

### ▶️ La distancia de cada equipamiento al efector más cercano

In [ ]:
comprobar_metrico(equip_en_barrio, "equipamientos")
comprobar_metrico(salud_m, "efectores de salud")

cercano = gpd.sjoin_nearest(
    equip_en_barrio,
    salud_m[["nombre", "geometry"]].rename(columns={"nombre": "efector_cercano"}),
    distance_col="dist_salud_m",
).drop(columns="index_right")

cercano[["name", "amenity", "barrio", "efector_cercano", "dist_salud_m"]].head()

### 👀 Verificación — el gradiente tiene que verse

Si `sjoin_nearest` funcionó, los puntos cercanos a un efector tienen que ser claros y los
lejanos oscuros: el mapa debería mostrar un halo alrededor de cada cruz roja.

In [ ]:
fig, eje = plt.subplots(figsize=(10, 8))

barrios_m.plot(ax=eje, facecolor="#f7f7f7", edgecolor="black", linewidth=1.5)
cercano.plot(ax=eje, column="dist_salud_m", cmap="YlOrRd", markersize=14,
             legend=True, legend_kwds=dict(label="Distancia al efector más cercano (m)",
                                           shrink=0.6))
salud_m.plot(ax=eje, color="black", marker="P", markersize=90)

eje.set_title("Cada equipamiento, coloreado por su distancia al efector de salud más cercano")
eje.set_axis_off()
plt.show()

El halo está: alrededor de cada cruz negra los puntos son amarillos, y se van oscureciendo hacia
los bordes. La operación hizo lo que decía.

### ▶️ De la distancia individual a la variable del barrio

`sjoin_nearest` devolvió una distancia por cada uno de los 1.769 equipamientos. Para la tabla de
análisis necesitamos **un número por barrio**: lo resume un `groupby`.

In [ ]:
resumen_dist = (cercano.groupby("barrio")["dist_salud_m"]
                .agg(["count", "mean", "median", "max"])
                .round(0))
resumen_dist

### 👀 La distribución completa, no sólo el promedio

Dos barrios pueden tener la misma media y distribuciones muy distintas. El histograma lo muestra.

In [ ]:
fig, eje = plt.subplots(figsize=(9, 5))

for barrio, color in [("Recoleta", "#c44e52"), ("Villa Lugano", "#4c72b0")]:
    datos = cercano.loc[cercano["barrio"] == barrio, "dist_salud_m"]
    eje.hist(datos, bins=30, alpha=0.6, label=barrio, color=color, density=True)

eje.axvline(500, color="black", linestyle="--", linewidth=1)
eje.text(520, eje.get_ylim()[1] * 0.9, "500 m", fontsize=9)
eje.set_xlabel("Distancia al efector de salud más cercano (m)")
eje.set_ylabel("Densidad")
eje.set_title("Distribución de la distancia, por barrio")
eje.legend()
plt.tight_layout()
plt.show()

### ▶️ La misma medición, por tipo de equipamiento

La pregunta se afina: no es lo mismo que esté lejos un café que una escuela.

In [ ]:
(cercano[cercano["amenity"].isin(["school", "pharmacy", "kindergarten", "cafe"])]
 .pivot_table(index="amenity", columns="barrio", values="dist_salud_m", aggfunc="median")
 .round(0))

### 🔍 Interpretación

La distancia mediana de un equipamiento cualquiera al efector de salud más cercano es de **413
metros en Villa Lugano y 603 en Recoleta**. Coincide con lo que dio la cobertura, y eso es
importante: **dos operaciones distintas sobre las mismas capas llegan al mismo resultado.**
Cuando eso pasa, la conclusión se sostiene.

El histograma agrega algo que el promedio escondía: la distribución de Villa Lugano está
corrida a la izquierda y es más angosta —casi todo su equipamiento está entre 200 y 700 metros
de un CeSAC—, mientras que la de Recoleta tiene una cola larga que llega a los 2,5 km.

Y la tabla por tipo es la que más sirve para un informe: **las escuelas de Villa Lugano están a
342 metros medianos de un efector de salud, y las de Recoleta a 559**. Para pensar programas de
salud escolar, ése es el número, no el promedio general.

---

## 11. La tabla de variables territoriales

### ▶️ Armarla

Todo el trabajo de hoy, en la forma en que sirve: una fila por unidad de análisis, una columna
por variable, la unidad de medida en el nombre.

In [ ]:
tabla = barrios_m[["barrio", "superficie_km2", "cobertura_500m_perc"]].copy()

tabla["efectores_salud"]   = tabla["barrio"].map(salud["barrio"].value_counts())
tabla["equipamientos"]     = tabla["barrio"].map(conteo)
tabla["farmacias_por_km2"] = tabla["barrio"].map(densidad["pharmacy_por_km2"])
tabla["escuelas_por_km2"]  = tabla["barrio"].map(densidad["school_por_km2"])
tabla["dist_salud_m"]      = tabla["barrio"].map(resumen_dist["median"])

tabla.round(1)

### 👀 El perfil de cada barrio, de un vistazo

Las variables están en unidades distintas —km², porcentajes, metros—, así que para compararlas
en un mismo gráfico hay que llevarlas a una escala común. Un **min–max** las pone a todas entre
0 y 1.

In [ ]:
variables = ["farmacias_por_km2", "escuelas_por_km2", "cobertura_500m_perc", "dist_salud_m"]

normalizada = tabla.set_index("barrio")[variables]
normalizada = (normalizada - normalizada.min()) / (normalizada.max() - normalizada.min())

eje = normalizada.T.plot.barh(figsize=(9, 5), color=["#c44e52", "#4c72b0"], width=0.75)
eje.set_xlabel("Valor normalizado (0 = el menor de los dos, 1 = el mayor)")
eje.set_ylabel("")
eje.set_title("Perfil territorial de los dos barrios")
eje.legend(title="")
eje.grid(axis="x", linestyle=":", alpha=0.6)
plt.tight_layout()
plt.show()

> ⚠️ Con dos unidades, un min–max siempre da 0 y 1: el gráfico sirve para leer **el sentido** de
> cada contraste, no su magnitud. Con veinte barrios la normalización empieza a ser informativa,
> y es lo que vamos a usar en la Clase 7 para construir un índice.

### ▶️ Exportarla

Tres formatos, tres usos. El GeoPackage conserva la geometría y es con lo que se sigue
trabajando; el CSV es la tabla para el informe; el PNG es la figura.

In [ ]:
salida = barrios[["barrio", "geometry"]].merge(tabla, on="barrio")

salida.to_file("variables_territoriales_barrios.gpkg", driver="GPKG")
tabla.round(2).to_csv("variables_territoriales_barrios.csv", index=False)

print("Guardados:")
print("  variables_territoriales_barrios.gpkg  — con geometría, para seguir trabajando")
print("  variables_territoriales_barrios.csv   — la tabla, para el informe")

> ⚠️ En Colab estos archivos quedan en el disco temporal de la sesión y **se borran al
> cerrarla**. Para conservarlos hay que bajarlos con el panel de archivos de la izquierda, o
> montar Google Drive.

---

## 12. Apéndice — de la dirección al punto

### 🧭 Concepto

Este bloque no construye ninguna variable: resuelve el paso **anterior** a todo lo que hicimos
hoy, y es el que más falta les va a hacer en el trabajo final si sus datos vienen de una
encuesta o de un registro administrativo.

**Geocodificar** es convertir una dirección en coordenadas. Va en los dos sentidos:

| | De | A | Para qué |
|---|---|---|---|
| **Directa** | una dirección | un punto | Poner en el mapa una base de domicilios |
| **Inversa** | un punto | una dirección | Ponerle domicilio a un GPS, a una foto, a un registro |

Usamos **Nominatim**, el geocodificador de OpenStreetMap: gratuito, sin registro, un pedido por
segundo. Hacemos el viaje completo —de punto a dirección y de vuelta a punto— porque eso nos
deja **medir** cuánto nos alejamos del lugar original.

### ▶️ Paso 1 — De la coordenada a la dirección

`RateLimiter` respeta el segundo de espera que exige el servicio. Son 13 consultas: unos 15
segundos.

In [ ]:
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

nominatim = Nominatim(user_agent="sig-unab-2026-clase6")
inverso = RateLimiter(nominatim.reverse, min_delay_seconds=1.1)

respuestas = [inverso((punto.y, punto.x)) for punto in salud.geometry]

salud["calle"]  = [r.raw["address"].get("road") for r in respuestas]
salud["altura"] = [r.raw["address"].get("house_number") for r in respuestas]
salud["tiene_direccion"] = salud["calle"].notna() & salud["altura"].notna()

print(f"Con calle: {salud['calle'].notna().sum()} de {len(salud)}")
print(f"Con dirección completa: {salud['tiene_direccion'].sum()} de {len(salud)}")
salud[["barrio", "nombre", "calle", "altura"]]

### ▶️ Paso 2 — De la dirección de vuelta al punto

Cuanto más contexto tenga la consulta, menos ambigüedad: "Gallo 1330" existe en muchas ciudades
del país.

In [ ]:
directo = RateLimiter(nominatim.geocode, min_delay_seconds=1.1)

salud["consulta"] = (salud["calle"] + " " + salud["altura"].astype(str)
                     + ", " + salud["barrio"]
                     + ", Ciudad Autónoma de Buenos Aires, Argentina")

hallados = [directo(c) if tiene else None
            for c, tiene in zip(salud["consulta"], salud["tiene_direccion"])]

salud["lon_geo"] = [r.longitude if r else None for r in hallados]
salud["lat_geo"] = [r.latitude  if r else None for r in hallados]

print(f"Resolvió {salud['lat_geo'].notna().sum()} de {salud['tiene_direccion'].sum()}")

### ▶️ Paso 3 — Cuánto nos movimos

In [ ]:
geocodificados = gpd.GeoDataFrame(
    salud, geometry=gpd.points_from_xy(salud["lon_geo"], salud["lat_geo"]), crs=GEOGRAFICO)

d = salud_m.geometry.distance(geocodificados.to_crs(METRICO).geometry)
salud["error_m"] = d.where(d.notna()).round()

salud.groupby("barrio")["error_m"].agg(["count", "median", "max"]).round(1)

### 👀 El punto original y el geocodificado

A esta escala hay que hacer zoom para ver la diferencia, que es justamente la conclusión.

In [ ]:
fig, eje = plt.subplots(figsize=(9, 8))

barrios_m.plot(ax=eje, facecolor="#f7f7f7", edgecolor="black", linewidth=1.2)
salud_m.plot(ax=eje, color="crimson", markersize=70, label="Original (IGN)")
geocodificados.to_crs(METRICO).plot(ax=eje, color="navy", marker="x", markersize=70,
                                    label="Geocodificado (Nominatim)")

eje.legend(loc="upper right")
eje.set_title("Punto original y punto recuperado desde la dirección")
eje.set_axis_off()
plt.show()

### 🔍 Interpretación — funciona, y el error es medible

Las diez direcciones completas volvieron a coordenada: **10 de 10**. El error mediano es de
**4,5 metros en Villa Lugano y 36,5 en Recoleta**, con un máximo de 89.

Para cualquier variable de las que construimos hoy eso es despreciable: 89 metros es menos de
una cuadra y nuestras unidades miden kilómetros. En el mapa los pares de símbolos están
prácticamente encimados.

Dos cosas que sí importan.

**El error es más chico en Villa Lugano que en Recoleta.** Un CeSAC ocupa un edificio sobre una
calle, con una puerta y una altura; el Hospital de Clínicas ocupa media manzana y tiene entradas
por tres calles distintas. Cuanto más grande el establecimiento, más arbitrario es cuál es "su"
punto. Es un límite del modelo de datos, no del geocodificador: **representamos con un punto
algo que no es un punto**.

**Y la regla práctica:** se geocodifica **por dirección**, no por nombre. La dirección es un
dato estructurado y estandarizado; el nombre institucional es texto libre, y el servicio sólo
resuelve los pocos establecimientos célebres que alguien se tomó el trabajo de cargar.

> 🤖 **Actividad de IA.** Pedile a un asistente la dirección del CeSAC Nº 29 de Villa Lugano.
> Compará con la que devolvió la geocodificación inversa. Si coinciden, geocodificá la que te dio
> la IA y medí a cuántos metros cae del punto del IGN. Un dato plausible y bien formateado no es
> un dato verificado: el que verificaste, sí.

---

## 13. 🧪 Tu turno

Repetí el análisis cambiando **el equipamiento de referencia**: en lugar de los efectores de
salud del IGN, usá las **escuelas** de OpenStreetMap, que dieron densidad idéntica en los dos
barrios.

La celda de abajo tiene el esqueleto de las tres operaciones. **Agregale el mapa**: sin la
verificación visual no sabés si el resultado es el que creés.

In [ ]:
# 1 · Elegí el equipamiento de referencia y el radio
TIPO  = "school"      # probá también: "pharmacy", "kindergarten", "bank"
RADIO = 500           # en metros

referencia = equip_en_barrio[equip_en_barrio["amenity"] == TIPO]

# 2 · Cobertura: buffer + disolver + intersección
zona = referencia.buffer(RADIO).union_all()
resultado = barrios_m[["barrio", "superficie_km2"]].copy()
resultado["cubierto_perc"] = [round(100 * g.intersection(zona).area / g.area, 1)
                              for g in barrios_m.geometry]

# 3 · Distancia: al más cercano
otros = equip_en_barrio[equip_en_barrio["amenity"] != TIPO]
dist = gpd.sjoin_nearest(otros, referencia[["geometry"]], distance_col="dist_m")
resultado["dist_mediana_m"] = resultado["barrio"].map(
    dist.groupby("barrio")["dist_m"].median().round(0))

resultado

In [ ]:
# 4 · 👀 El mapa. Completá: cubierto en verde, sin cubrir en rojo, referencia en negro.
fig, eje = plt.subplots(figsize=(9, 8))

# ...

eje.set_title(f"Cobertura de {TIPO} a {RADIO} m")
eje.set_axis_off()
plt.show()

**Las preguntas, para responder por escrito:**

1. **¿Qué barrio queda mejor cubierto?** ¿Coincide con lo que mostraba la densidad por km² del
   bloque 8? Si dice otra cosa, ¿por qué puede ser?
2. **Cambiá el radio** a 300 y a 1.000 metros y rehacé el gráfico de sensibilidad del bloque 9.
   ¿Las curvas se cruzan? ¿Qué te dice eso sobre lo defendible que es tu conclusión?
3. **Cobertura y distancia**, ¿dan la misma respuesta? Cuando dos operaciones distintas coinciden
   el resultado se sostiene mejor; si no coinciden, hay algo que explicar.
4. **Mirá el mapa que hiciste.** ¿Hay alguna zona cubierta que te llame la atención? ¿Algún punto
   donde el resultado no se parezca a lo que sabés del barrio?

**Entrega:** media carilla, la tabla de resultados y los dos mapas, antes del próximo encuentro.

---

## 14. 📋 El trabajo final

A partir de hoy tenés todo lo necesario para empezar el trabajo con el que se aprueba el
seminario. Las cuatro familias de operaciones de esta clase —transformar, relacionar, superponer
y medir— son, literalmente, la caja de herramientas con la que vas a construir tus variables.

La consigna completa está en
[`TRABAJO_FINAL.md`](https://github.com/renzoepolo/sig-ciencias-sociales/blob/main/TRABAJO_FINAL.md)
y también en el aula virtual.

En una línea: **una notebook de Colab que corra de principio a fin, con una pregunta territorial,
al menos dos fuentes abiertas, dos variables territoriales construidas, dos mapas terminados y
una sección de limitaciones.**

Lo único que conviene hacer esta semana es lo primero: **elegir la pregunta y la unidad de
análisis**, y chequear que existan los datos. Traelo a la Clase 7 y lo miramos.

---

## 15. Cierre

### El recorrido de hoy

| Familia | Operación | Qué pregunta responde | La variable que salió |
|---|---|---|---|
| — | `merge` por clave | ¿Cómo junto la tabla con la geometría? | `perc_65_mas` |
| **Transformar** | `centroid`, `buffer`, `simplify`, `union_all` | ¿Qué forma necesito para medir lo que quiero medir? | — |
| **Relacionar** | `sjoin` + `groupby` | ¿Qué hay adentro de cada unidad? | `farmacias_por_km2` |
| **Superponer** | `intersection`, `difference` | ¿Qué parte de la unidad está dentro de la zona? | `cobertura_500m_perc` |
| **Medir** | `sjoin_nearest` | ¿A qué distancia está lo más cercano? | `dist_salud_m` |

### La idea para llevarse

Una variable territorial **no se observa: se construye**, y cada una lleva adentro decisiones que
no quedan escritas en la columna. El radio de 500 metros, el predicado `within`, la elección de
dividir por superficie y no por población: nada de eso está en el nombre `cobertura_500m_perc`,
y todo eso cambia el número.

Por eso la clase verificó cada paso con un mapa. **Un resultado que no miraste no es un
resultado**: es un número que salió. Las dos veces que hoy el mapa dijo algo que la tabla no
decía —el halo de `sjoin_nearest`, las curvas que no se cruzan— fueron las dos veces que la
conclusión pasó de plausible a defendible.

### Glosario de la clase

| Término | Definición |
|---|---|
| **Variable territorial** | Atributo de una unidad del espacio que se calcula a partir de la posición relativa de dos o más capas. |
| **Unión por clave** (`merge`) | Emparejamiento de dos tablas por una columna común. No usa la geometría. |
| **Unión espacial** (`sjoin`) | Emparejamiento de dos capas por su posición relativa. No necesita atributos en común. |
| **Predicado espacial** | La relación que la unión espacial busca: `within`, `intersects`, `contains`. |
| **Centroide** | Centro de masa de una geometría. No siempre cae adentro de ella. |
| **Área de influencia** (*buffer*) | Zona que rodea a una geometría hasta una distancia dada. |
| **Envolvente convexa** (*convex hull*) | El polígono convexo más chico que contiene a la geometría. |
| **Simplificación** | Reducción de la cantidad de vértices conservando la forma general, con una tolerancia. |
| **Disolver** (`union_all`) | Fundir varias geometrías en una, eliminando superposiciones. |
| **Superposición** (*overlay*) | Operación que corta dos capas entre sí: `intersection`, `difference`, `union`. |
| **Cobertura** | Porcentaje de la superficie de una unidad que cae dentro de un área de influencia. |
| **Vecino más cercano** (`sjoin_nearest`) | Para cada elemento de una capa, el más próximo de otra, y su distancia. |
| **Geocodificación** | Conversión de una dirección en coordenadas, o a la inversa. |

### Tres preguntas para autoevaluarse

1. Calculaste el porcentaje de cobertura de un barrio y te dio 137 %. ¿Qué paso te faltó?
2. Querés medir la distancia de cada radio censal al hospital más cercano. ¿Qué transformación
   de geometría necesitás antes, y qué supuesto estás haciendo al usarla?
3. Un `sjoin` con `within` te dejó 40 puntos sin asignar de 1.000. ¿Cuáles son las dos
   explicaciones posibles, y cómo las distinguís?

### Qué viene en la Clase 7

Hoy medimos en **línea recta**. Nadie camina en línea recta.

La semana que viene calculamos la distancia y el tiempo **por la red de calles**, y comparamos:
vamos a ver cuánto sobreestima el círculo de 500 metros que dibujamos hoy. Y resolvemos el
problema que quedó abierto en el bloque 9: **cómo pasar una variable de una división territorial
a otra** —de radios censales a barrios— para poder calcular la cobertura sobre población y no
sobre superficie.

---

## 16. Referencias

- de Smith, M. J., Goodchild, M. F. y Longley, P. A. (2018). *Geospatial Analysis: A
  Comprehensive Guide to Principles, Techniques and Software Tools* (6.ª ed.), caps. 4 y 7.
- Rey, S., Arribas-Bel, D. y Wolf, L. J. (2023). *Geographic Data Science with Python*,
  cap. 8 "Spatial Feature Engineering". CRC Press.
- INDEC (2023). *Censo Nacional de Población, Hogares y Viviendas 2022. Resultados definitivos*.
- Instituto Geográfico Nacional (2024). *Capas de Sistema de Información Geográfica — Salud*.
  Geoservicio WFS.
- Ministerio de Salud del Gobierno de la Ciudad de Buenos Aires. *Centros de Salud y Acción
  Comunitaria (CeSAC)*.